# QuestMasters — DM Orchestrator en Colab

Corre el modelo Gemma 4 E4B con LoRAs en la GPU de Colab (T4, 15GB VRAM).
Tu backend local se conecta a este servidor via ngrok.

**Pasos:**
1. Configura los secretos en el panel izquierdo (icono de llave):
   - `HF_TOKEN` — tu token de HuggingFace
   - `NGROK_TOKEN` — tu token de ngrok (gratis en https://ngrok.com)
2. Ejecuta todas las celdas
3. Copia la URL de ngrok que aparece al final
4. En tu `.dev.vars` del backend, pon:
   ```
   DM_MODEL_ENDPOINT_MAS=<url_ngrok>
   DM_MODEL_ENDPOINT_MONOLITHIC=<url_ngrok>
   ```
5. Reinicia el backend (`bun dev`)

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes safetensors \
    huggingface_hub fastapi uvicorn pyngrok lightrag-hku langgraph \
    networkx chromadb pydantic

In [ ]:
import os
from google.colab import userdata

# ── Secretos de Colab (configuralos en el icono de llave del panel izquierdo) ──
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["DM_MODE"] = "local"
os.environ["DM_LOCAL_PORT"] = "8000"
NGROK_TOKEN = userdata.get("NGROK_TOKEN")

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

## 1. Clonar el orchestrator desde el repo

In [ ]:
%%bash
if [ ! -d "/content/orchestrator" ]; then
  mkdir -p /content/orchestrator
fi

In [ ]:
%%writefile /content/orchestrator/config.py
import os
from pathlib import Path

MODE = "local"
HF_TOKEN = os.environ.get("HF_TOKEN", "")
BASE_MODEL_ID = "google/gemma-4-e4b-it"
BASE_MODEL_LOCAL = Path("/content/base_model")
LORA_BASE_PATH = Path("/content/lora_weights")
SESSIONS_DIR = Path("/content/sessions")
LORA_REPOS = {
    "narrator": "Questmasters/e4b_narrador",
    "arbiter": "Questmasters/e4b_reglas",
    "npc": "Questmasters/e4b_npc",
    "state": "Questmasters/e4b_estado",
    "monolithic": "Questmasters/e4b_monolitico",
}

## 2. Cargar modelo + LoRAs (4-bit, todo en GPU)

In [ ]:
import torch
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from pathlib import Path

HF_TOKEN = os.environ["HF_TOKEN"]
BASE_MODEL_ID = "google/gemma-4-e4b-it"
LORA_REPOS = {
    "narrator": "Questmasters/e4b_narrador",
    "arbiter": "Questmasters/e4b_reglas",
    "npc": "Questmasters/e4b_npc",
    "state": "Questmasters/e4b_estado",
    "monolithic": "Questmasters/e4b_monolitico",
}
LORA_BASE_PATH = Path("/content/lora_weights")

# Descargar LoRAs
for name, repo_id in LORA_REPOS.items():
    local_dir = LORA_BASE_PATH / name
    if not local_dir.exists():
        print(f"Descargando LoRA '{name}' ...")
        snapshot_download(repo_id=repo_id, local_dir=str(local_dir), token=HF_TOKEN)
    else:
        print(f"LoRA '{name}' en cache")

# Cargar modelo base en 4-bit
print("\nCargando modelo base (4-bit NF4)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map={"":0},
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
print("Modelo base cargado.")

# Cargar todos los LoRA adapters
first_name, *rest_names = list(LORA_REPOS.keys())
print(f"Cargando LoRA '{first_name}'...")
model = PeftModel.from_pretrained(base, str(LORA_BASE_PATH / first_name), adapter_name=first_name)
for name in rest_names:
    print(f"Cargando LoRA '{name}'...")
    model.load_adapter(str(LORA_BASE_PATH / name), adapter_name=name)

print("\nTodos los LoRAs cargados!")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f"VRAM usada: {(total - free) / 1024**3:.1f} GB / {total / 1024**3:.1f} GB")

## 3. Levantar servidor FastAPI + ngrok

In [ ]:
%%writefile /content/orchestrator/schemas.py
from __future__ import annotations
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field


class CharacterSnapshot(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    name: str
    race: str = ""
    class_name: str = Field(default="", alias="class")
    background: str = ""
    description: str = ""


class DmModelRequest(BaseModel):
    session_id: str
    architecture_type: Literal["monolithic", "mas"]
    model_id: str
    campaign_prompt: str
    characters: list[CharacterSnapshot]
    conversation_history: list[dict]
    player_input: str | None = None
    current_memory_snapshot: dict


class DeltaChunk(BaseModel):
    type: Literal["delta"] = "delta"
    content: str


class MemorySnapshot(BaseModel):
    l2_episode_ids: list[str] = []
    l3_entity_ids: list[str] = []


class UsageStats(BaseModel):
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int


class MetadataChunk(BaseModel):
    type: Literal["metadata"] = "metadata"
    memory_snapshot: MemorySnapshot
    usage: UsageStats
    latency_ms: int


class DoneChunk(BaseModel):
    type: Literal["done"] = "done"


class ErrorChunk(BaseModel):
    type: Literal["error"] = "error"
    message: str


SseChunk = DeltaChunk | MetadataChunk | DoneChunk | ErrorChunk

In [ ]:
import asyncio
import json
import re
import time
import queue
import threading
from concurrent.futures import ThreadPoolExecutor
from typing import Any, AsyncGenerator

!pip install -q nest_asyncio
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from transformers import TextIteratorStreamer

import sys
sys.path.insert(0, "/content/orchestrator")
from schemas import (
    DeltaChunk, DmModelRequest, DoneChunk, ErrorChunk,
    MetadataChunk, MemorySnapshot, UsageStats, SseChunk,
)

app = FastAPI(title="QuestMasters DM Orchestrator (Colab)")
_executor = ThreadPoolExecutor(max_workers=1)

_CAMEL_RE_1 = re.compile(r"(.)([A-Z][a-z]+)")
_CAMEL_RE_2 = re.compile(r"([a-z0-9])([A-Z])")

SYSTEM_PROMPT = (
    "You are a Dungeon Master for a D&D 5e session. "
    "Think and reason internally in English using your training knowledge, "
    "but ALWAYS respond to the player in Spanish. "
    "Be descriptive, atmospheric, and follow D&D 5e rules naturally in your narration. "
    "Keep responses concise: 2-4 paragraphs maximum. "
    "Never use emojis. Never mix languages. Always end by asking the player what they do next."
)

def _camel_to_snake(name: str) -> str:
    return _CAMEL_RE_2.sub(r"\1_\2", _CAMEL_RE_1.sub(r"\1_\2", name)).lower()

def _convert_keys(obj):
    if isinstance(obj, dict):
        return {_camel_to_snake(k): _convert_keys(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_convert_keys(i) for i in obj]
    return obj


def _build_messages(request: DmModelRequest) -> list[dict]:
    chars = []
    for c in request.characters:
        parts = [c.name]
        if c.race:
            parts.append(c.race)
        if c.class_name:
            parts.append(c.class_name)
        chars.append(" - ".join(parts))
    characters = ", ".join(chars) if chars else "Sin personajes definidos"

    campaign = request.campaign_prompt.strip() if request.campaign_prompt else ""
    player = request.player_input or ""

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if not player:
        if campaign:
            user_msg = (
                f"Campaign: {campaign}\n"
                f"Characters: {characters}\n"
                f"Narrate the opening scene of this adventure in 2-3 paragraphs. "
                f"Present the world and the characters' situation. "
                f"End by asking what the players do. Respond in Spanish. No emojis."
            )
        else:
            user_msg = (
                f"Characters: {characters}\n"
                f"Create a fantasy world and narrate the opening scene in 2-3 paragraphs. "
                f"End by asking what the players do. Respond in Spanish. No emojis."
            )
    else:
        history = "\n".join(
            f"{m['role'].upper()}: {m['content']}"
            for m in request.conversation_history[-6:]
        )
        user_msg = (
            f"{history}\n"
            f"Player: {player}\n"
            f"Respond as DM in 2-3 paragraphs in Spanish. No emojis. "
            f"End by asking the player what they do next."
        )

    messages.append({"role": "user", "content": user_msg})
    return messages


def _clean_response(text: str) -> str:
    """Remove emojis and non-latin characters from response."""
    import unicodedata
    cleaned = []
    for char in text:
        cat = unicodedata.category(char)
        if cat.startswith("So") or cat.startswith("Cs"):
            continue
        if ord(char) > 0x2000 and not char in "—–''""…·":
            continue
        cleaned.append(char)
    return "".join(cleaned)


def _generate_to_queue(request: DmModelRequest, q: queue.Queue) -> None:
    import traceback
    try:
        adapter = "monolithic" if request.architecture_type == "monolithic" else "narrator"
        model.set_adapter(adapter)

        messages = _build_messages(request)
        input_ids = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,
            return_dict=False,
        ).to(model.device)

        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

        gen_kwargs = {
            "input_ids": input_ids,
            "max_new_tokens": 300,
            "temperature": 0.6,
            "top_k": 40,
            "top_p": 0.9,
            "do_sample": True,
            "streamer": streamer,
            "repetition_penalty": 1.3,
        }
        t = threading.Thread(target=model.generate, kwargs=gen_kwargs, daemon=True)
        t.start()

        t_start = time.monotonic()
        for token in streamer:
            cleaned = _clean_response(token)
            if cleaned:
                q.put(DeltaChunk(content=cleaned))

        latency_ms = int((time.monotonic() - t_start) * 1000)
        q.put(MetadataChunk(
            memory_snapshot=MemorySnapshot(),
            usage=UsageStats(prompt_tokens=0, completion_tokens=0, total_tokens=0),
            latency_ms=latency_ms,
        ))
        q.put(DoneChunk())
        q.put(None)
    except Exception as exc:
        exc.__traceback_str__ = traceback.format_exc()
        q.put(exc)


def _chunk_to_sse(chunk: SseChunk, model_id: str) -> str:
    if isinstance(chunk, DeltaChunk):
        payload = {"type": "delta", "delta": chunk.content}
    elif isinstance(chunk, MetadataChunk):
        payload = {
            "type": "metadata",
            "metadata": {
                "memorySnapshot": chunk.memory_snapshot.model_dump(),
                "narrativeNotesDelta": [],
                "usage": {
                    "inputTokens": chunk.usage.prompt_tokens,
                    "outputTokens": chunk.usage.completion_tokens,
                },
                "latencyMs": chunk.latency_ms,
                "modelId": model_id,
            },
        }
    elif isinstance(chunk, DoneChunk):
        payload = {"type": "done"}
    elif isinstance(chunk, ErrorChunk):
        payload = {"type": "error", "error": chunk.message}
    else:
        return ""
    return f"data: {json.dumps(payload)}\n\n"


@app.get("/health")
async def health():
    return {"status": "ok", "mode": "colab", "gpu": torch.cuda.get_device_name(0)}


@app.post("/generate")
async def generate(body: dict[str, Any]) -> StreamingResponse:
    session_id = body.get("sessionId", body.get("session_id", "?"))
    print(f"Request recibido | session_id={session_id}")

    try:
        snake_body = _convert_keys(body)
        request = DmModelRequest(**snake_body)
    except Exception as exc:
        error = json.dumps({"type": "error", "error": f"Input invalido: {exc}"})
        return StreamingResponse(iter([f"data: {error}\n\n"]), media_type="text/event-stream")

    chunk_queue: queue.Queue = queue.Queue()
    loop = asyncio.get_event_loop()
    loop.run_in_executor(_executor, _generate_to_queue, request, chunk_queue)

    async def stream() -> AsyncGenerator[str, None]:
        while True:
            try:
                item = chunk_queue.get_nowait()
            except queue.Empty:
                yield ": keepalive\n\n"
                await asyncio.sleep(2)
                continue
            if item is None:
                break
            if isinstance(item, Exception):
                tb = getattr(item, "__traceback_str__", str(item))
                print(f"Error:\n{tb}")
                yield f"data: {json.dumps({'type': 'error', 'error': str(item)})}\n\n"
                break
            yield _chunk_to_sse(item, request.model_id)

    return StreamingResponse(stream(), media_type="text/event-stream")

print("FastAPI app definida.")

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
print("=" * 60)
print(f"SERVIDOR PUBLICO: {public_url}")
print("=" * 60)
print(f"\nPon esto en tu .dev.vars del backend:")
print(f"  DM_MODEL_ENDPOINT_MAS={public_url}")
print(f"  DM_MODEL_ENDPOINT_MONOLITHIC={public_url}")
print(f"  DM_USE_RUNPOD=false")

In [ ]:
# Iniciar servidor (esta celda bloquea - el servidor queda corriendo)
import uvicorn

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()